# Pixels to Predictions — Colab Solution

**Workflow:**
1. Mount Google Drive (saves checkpoints, survives session disconnects)
2. Copy competition data from Drive to local Colab storage (faster I/O during training)
3. QLoRA fine-tune SmolVLM-500M (<=5M trainable params)
4. Log-likelihood inference (more reliable than text generation + parsing)
5. Download submission.csv and upload to Kaggle

**Before running:**
- Runtime -> Change runtime type -> **T4 GPU**
- Upload your entire `Data/` folder to Google Drive (e.g. `MyDrive/competition_data/`)
  - The folder should contain: `train.csv`, `val.csv`, `test.csv`, and `train/`, `val/`, `test/` image subdirectories
- Update `DRIVE_DATA_DIR` in Cell 2 to match where you uploaded it

In [1]:
# ── Cell 0: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully")

Mounted at /content/drive
Google Drive mounted successfully


In [2]:
# ── Cell 1: Install packages ──────────────────────────────────────────────────
!pip install -q transformers==4.51.3 peft==0.14.0 bitsandbytes==0.45.5 accelerate==1.6.0 kaggle pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.9 MB/s eta 0:00:00


In [3]:
# ── Cell 2: Copy competition data from Google Drive to local Colab storage ────
import os, shutil

# Set this to wherever you uploaded your Data/ folder in Google Drive.
# e.g. if you put it at MyDrive/competition_data/, use that path below.
DRIVE_DATA_DIR = "/content/drive/MyDrive/competition_data"   # <-- update if needed

DATA_DEST = "/content/data"

if not os.path.exists(DATA_DEST):
    print(f"Copying data from Drive to local storage (faster I/O for training)...")
    shutil.copytree(DRIVE_DATA_DIR, DATA_DEST)
    print("Done.")
else:
    print("Data already in local storage, skipping copy.")

!ls {DATA_DEST}

Copying data from Drive to local storage (faster I/O for training)...
Done.
images	sample_submission.csv  test.csv  train.csv  val.csv


In [4]:
# ── Cell 3: Imports & global config ──────────────────────────────────────────
import json, random, math
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path("/content/data")                               # extracted competition data
DRIVE_DIR = Path("/content/drive/MyDrive/smolvlm_competition") # Drive output folder
CKPT_DIR  = DRIVE_DIR / "best_ckpt"                            # best checkpoint (persists across sessions)
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"  # Colab has internet access

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE     = 384
EPOCHS       = 3
LR           = 2e-4
BATCH_SIZE   = 2
GRAD_ACCUM   = 8
MAX_SEQ_LEN  = 2048
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]
# Param count: 4 x 32 layers x 2 x 16 x 960 = 3,932,160 (< 5M limit)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [5]:
# ── Cell 4: Load data + prompt builder ───────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(
    row: pd.Series,
    include_answer: bool = False,
    use_solution: bool = False,
) -> str:
    """
    use_solution=True: inject the solution field as context (train/val only; not in test set)
    include_answer=True: append the correct answer letter (used to compute training loss)
    """
    parts = []
    for field in ("lecture", "hint"):
        val = row.get(field, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())

    if use_solution:
        sol = row.get("solution", "")
        if pd.notna(sol) and str(sol).strip():
            parts.append(str(sol).strip())

    context_str = "\n".join(parts)
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row["choices"])
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer with a single letter.\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt

print(build_prompt(train_df.iloc[0], include_answer=True, use_solution=True))

Train: 3,109 | Val: 1,048 | Test: 1,008
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or c

In [6]:
# ── Cell 5: Dataset ───────────────────────────────────────────────────────────
class SciQADataset(Dataset):
    def __init__(
        self, df, data_dir, processor,
        img_size=384, is_train=True, use_solution=False, max_seq_len=512,
    ):
        self.df           = df.reset_index(drop=True)
        self.data_dir     = data_dir
        self.processor    = processor
        self.img_size     = img_size
        self.is_train     = is_train
        self.use_solution = use_solution
        self.max_seq_len  = max_seq_len

    def __len__(self):
        return len(self.df)

    def _load_image(self, rel_path):
        return (
            Image.open(self.data_dir / rel_path)
            .convert("RGB")
            .resize((self.img_size, self.img_size), Image.BICUBIC)
        )

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = self._load_image(row["image_path"])

        if self.is_train:
            text = build_prompt(row, include_answer=True, use_solution=self.use_solution)
            enc = self.processor(
                text=[text], images=[image], return_tensors="pt",
            )
            input_ids      = enc["input_ids"].squeeze(0)
            attention_mask = enc["attention_mask"].squeeze(0)
            pixel_values   = enc["pixel_values"].squeeze(0)

            pad_id = self.processor.tokenizer.pad_token_id
            non_pad_positions = (input_ids != pad_id).nonzero(as_tuple=True)[0]
            last_content_idx  = non_pad_positions[-1]
            labels = torch.full_like(input_ids, -100)
            labels[last_content_idx] = input_ids[last_content_idx]

            return {
                "input_ids": input_ids, "attention_mask": attention_mask,
                "pixel_values": pixel_values, "labels": labels,
            }
        else:
            return {
                "id":      row["id"],
                "image":   image,
                "prompt":  build_prompt(row, include_answer=False, use_solution=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

In [7]:
# ── Cell 6: Load model + apply QLoRA ─────────────────────────────────────────
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
)
model = get_peft_model(model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
assert trainable <= 5_000_000, f"ERROR: {trainable:,} exceeds the 5M parameter limit!"
print("Trainable parameter count is within the 5M limit")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable: 4,161,536 / 305,991,872 (1.36%)
Trainable parameter count is within the 5M limit


In [8]:
# ── Cell 7: DataLoader ────────────────────────────────────────────────────────
train_ds = SciQADataset(
    train_df, DATA_DIR, processor,
    img_size=IMG_SIZE, is_train=True, use_solution=True, max_seq_len=MAX_SEQ_LEN,
)
val_ds = SciQADataset(
    val_df, DATA_DIR, processor, img_size=IMG_SIZE, is_train=False,
)
test_ds = SciQADataset(
    test_df, DATA_DIR, processor, img_size=IMG_SIZE, is_train=False,
)

import torch.nn.functional as F

def collate_fn(batch):
    max_len = max(item["input_ids"].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id
    result = {"pixel_values": torch.stack([item["pixel_values"] for item in batch])}
    for key, pad_val in [("input_ids", pad_id), ("attention_mask", 0), ("labels", -100)]:
        result[key] = torch.stack([
            F.pad(item[key], (0, max_len - item[key].shape[0]), value=pad_val)
            for item in batch
        ])
    return result

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
    collate_fn=collate_fn,
)
print(f"train={len(train_ds)} | val={len(val_ds)} | test={len(test_ds)}")
print(f"Batches per epoch: {len(train_loader)}")

train=3109 | val=1048 | test=1008
Batches per epoch: 1555


In [9]:
# ── Cell 8: Inference helpers ─────────────────────────────────────────────────
@torch.no_grad()
def score_choices(model, processor, image, prompt, choices):
    """Log-likelihood scoring: one forward pass, argmax over answer letter logits."""
    enc = processor(text=[prompt], images=[image], return_tensors="pt")
    enc = {k: v.to(model.device) for k, v in enc.items()}
    logits = model(**enc).logits[0, -1, :]   # logits at last token position
    scores = []
    for i in range(len(choices)):
        token_id = processor.tokenizer.encode(
            f" {CHOICE_LETTERS[i]}", add_special_tokens=False
        )[-1]
        scores.append(logits[token_id].item())
    return int(np.argmax(scores))

@torch.no_grad()
def evaluate(model, ds, desc="eval", n_samples=None):
    model.eval()
    correct, total = 0, 0
    n = len(ds) if n_samples is None else min(n_samples, len(ds))
    for i in range(n):
        sample = ds[i]
        if sample["answer"] == -1:
            continue
        pred = score_choices(model, processor, sample["image"], sample["prompt"], sample["choices"])
        correct += int(pred == sample["answer"])
        total += 1
    acc = correct / total if total > 0 else 0.0
    print(f"{desc}: {correct}/{total} = {acc:.4f}")
    return acc

In [ ]:
# ── Cell 9: Training loop (checkpoints saved to Google Drive) ─────────────────
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)

total_steps  = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = int(0.10 * total_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss, n_steps = 0.0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        try:
            loss = model(**batch).loss / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM
        except RuntimeError as e:
            if "shape mismatch" in str(e) or "memory" in str(e).lower():
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                continue
            raise e

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            n_steps += 1

        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} | step {step+1}/{len(train_loader)} "
                  f"| loss {total_loss/n_steps:.4f} "
                  f"| lr {scheduler.get_last_lr()[0]:.2e}")

    val_acc  = evaluate(model, val_ds, desc=f"Epoch {epoch+1} val")
    avg_loss = total_loss / max(n_steps, 1)
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"{'='*60}\n")

    # Save to Google Drive so the checkpoint survives a Colab disconnect
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(str(CKPT_DIR))
        processor.save_pretrained(str(CKPT_DIR))
        print(f"  New best model saved to Google Drive: {CKPT_DIR}")
        print(f"  Val Acc = {val_acc:.4f}")

print(f"\nTraining complete. Best Val Accuracy: {best_val_acc:.4f}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  Epoch 1 | step 50/1555 | loss 9.4587 | lr 2.07e-05
  Epoch 1 | step 100/1555 | loss 8.3119 | lr 4.14e-05
  Epoch 1 | step 150/1555 | loss 7.6290 | lr 6.21e-05
  Epoch 1 | step 200/1555 | loss 6.6239 | lr 8.62e-05
  Epoch 1 | step 250/1555 | loss 6.2578 | lr 1.07e-04
  Epoch 1 | step 300/1555 | loss 5.7943 | lr 1.28e-04
  Epoch 1 | step 350/1555 | loss 5.4090 | lr 1.48e-04
  Epoch 1 | step 400/1555 | loss 4.9494 | lr 1.72e-04
  Epoch 1 | step 450/1555 | loss 4.8093 | lr 1.93e-04
  Epoch 1 | step 500/1555 | loss 4.6093 | lr 2.00e-04
  Epoch 1 | step 550/1555 | loss 4.3565 | lr 2.00e-04
  Epoch 1 | step 600/1555 | loss 4.0897 | lr 1.99e-04
  Epoch 1 | step 650/1555 | loss 3.8859 | lr 1.99e-04
  Epoch 1 | step 700/1555 | loss 3.7832 | lr 1.99e-04
  Epoch 1 | step 750/1555 | loss 3.6694 | lr 1.98e-04
  Epoch 1 | step 800/1555 | loss 3.4818 | lr 1.97e-04
  Epoch 1 | step 850/1555 | loss 3.3514 | lr 1.96e-04
  Epoch 1 | step 900/1555 | loss 3.2522 | lr 1.95e-04
  Epoch 1 | step 950/1555 | l

In [ ]:
# ── Cell 10: Reload best Phase 1 ckpt + Phase 2 warm-start on train+val ───────
from peft import PeftModel

PHASE2        = True   # enabled for final submission
PHASE2_EPOCHS = 3      # fewer than Phase 1 — no validation set to catch overfitting
PHASE2_LR     = 1e-4   # half of Phase 1 LR; warm-start needs smaller steps

# ── Step A: Reload best Phase 1 checkpoint ────────────────────────────────────
print("Reloading best Phase 1 checkpoint from Google Drive...")
base_inf    = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map="auto", low_cpu_mem_usage=True,
)
best_phase1 = PeftModel.from_pretrained(base_inf, str(CKPT_DIR))
best_phase1.eval()
print(f"Best Phase 1 checkpoint loaded from {CKPT_DIR}")

# ── Step B: Phase 2 — warm-start from best Phase 1, train on train+val ────────
if PHASE2:
    print("\nPhase 2: warm-start retraining on combined train+val...")
    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    trainval_ds = SciQADataset(
        trainval_df, DATA_DIR, processor,
        img_size=IMG_SIZE, is_train=True, use_solution=True, max_seq_len=MAX_SEQ_LEN,
    )
    trainval_loader = DataLoader(
        trainval_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
        collate_fn=collate_fn,
    )

    base2  = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, quantization_config=bnb_cfg, device_map="auto", low_cpu_mem_usage=True,
    )
    base2  = prepare_model_for_kbit_training(base2)
    model2 = PeftModel.from_pretrained(base2, str(CKPT_DIR), is_trainable=True)

    opt2   = AdamW([p for p in model2.parameters() if p.requires_grad], lr=PHASE2_LR, weight_decay=0.01)
    steps2 = math.ceil(len(trainval_loader) / GRAD_ACCUM) * PHASE2_EPOCHS
    warm2  = int(0.10 * steps2)
    sched2 = get_cosine_schedule_with_warmup(opt2, warm2, steps2)

    for epoch in range(PHASE2_EPOCHS):
        model2.train()
        total_loss2, n_steps2 = 0.0, 0
        opt2.zero_grad()
        for step, batch in enumerate(trainval_loader):
            batch = {k: v.to(model2.device) if torch.is_tensor(v) else v for k, v in batch.items()}
            loss  = model2(**batch).loss / GRAD_ACCUM
            loss.backward()
            total_loss2 += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(trainval_loader):
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model2.parameters() if p.requires_grad], 1.0
                )
                opt2.step(); sched2.step(); opt2.zero_grad()
                n_steps2 += 1
            if (step + 1) % 100 == 0:
                print(f"  Phase2 Ep{epoch+1} | step {step+1}/{len(trainval_loader)} "
                      f"| loss {total_loss2/n_steps2:.4f} | lr {sched2.get_last_lr()[0]:.2e}")
        avg = total_loss2 / max(n_steps2, 1)
        print(f"Phase 2 Epoch {epoch+1}/{PHASE2_EPOCHS} done | avg loss {avg:.4f}")

        # Save each epoch to its own folder (~16 MB each) so earlier epochs aren't overwritten
        ep_ckpt = DRIVE_DIR / f"phase2_ep{epoch+1}"
        model2.save_pretrained(str(ep_ckpt))
        processor.save_pretrained(str(ep_ckpt))
        print(f"  Saved Phase 2 Epoch {epoch+1} → {ep_ckpt}")

    FINAL_CKPT = DRIVE_DIR / "final_ckpt"
    model2.save_pretrained(str(FINAL_CKPT))
    processor.save_pretrained(str(FINAL_CKPT))
    inference_model = model2
    inference_model.eval()
    print(f"Phase 2 model saved to: {FINAL_CKPT}")
else:
    inference_model = best_phase1
    print("Phase 2 disabled. Using best Phase 1 checkpoint for inference.")

Reloading best Phase 1 checkpoint from Google Drive...
Best Phase 1 checkpoint loaded from /content/drive/MyDrive/smolvlm_competition/best_ckpt

Phase 2: warm-start retraining on combined train+val...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  Phase2 Ep1 | step 100/2079 | loss 0.2894 | lr 1.54e-05
  Phase2 Ep1 | step 200/2079 | loss 0.2688 | lr 3.21e-05
  Phase2 Ep1 | step 300/2079 | loss 0.2575 | lr 4.74e-05
  Phase2 Ep1 | step 400/2079 | loss 0.2969 | lr 6.41e-05
  Phase2 Ep1 | step 500/2079 | loss 0.3158 | lr 7.95e-05
  Phase2 Ep1 | step 600/2079 | loss 0.3381 | lr 9.62e-05
  Phase2 Ep1 | step 700/2079 | loss 0.3667 | lr 1.00e-04
  Phase2 Ep1 | step 800/2079 | loss 0.5135 | lr 9.98e-05
  Phase2 Ep1 | step 900/2079 | loss 0.4957 | lr 9.94e-05
  Phase2 Ep1 | step 1000/2079 | loss 0.4978 | lr 9.89e-05
  Phase2 Ep1 | step 1100/2079 | loss 0.4969 | lr 9.83e-05
  Phase2 Ep1 | step 1200/2079 | loss 0.5182 | lr 9.74e-05
  Phase2 Ep1 | step 1300/2079 | loss 0.5099 | lr 9.65e-05
  Phase2 Ep1 | step 1400/2079 | loss 0.4883 | lr 9.54e-05
  Phase2 Ep1 | step 1500/2079 | loss 0.4665 | lr 9.42e-05
  Phase2 Ep1 | step 1600/2079 | loss 0.4944 | lr 9.27e-05
  Phase2 Ep1 | step 1700/2079 | loss 0.5297 | lr 9.13e-05
  Phase2 Ep1 | step 180

In [ ]:
# ── Cell 10b: Phase 3 — continue from phase2_ep3 at very low LR ───────────────
from peft import PeftModel

PHASE3_EPOCHS = 3
PHASE3_LR     = 5e-5   # quarter of Phase 1 LR; fine-grained refinement

print("Phase 3: loading phase2_ep3 and continuing on train+val...")
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
trainval_ds = SciQADataset(
    trainval_df, DATA_DIR, processor,
    img_size=IMG_SIZE, is_train=True, use_solution=True, max_seq_len=MAX_SEQ_LEN,
)
trainval_loader = DataLoader(
    trainval_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
    collate_fn=collate_fn,
)

base3  = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map="auto", low_cpu_mem_usage=True,
)
base3  = prepare_model_for_kbit_training(base3)
model3 = PeftModel.from_pretrained(base3, str(DRIVE_DIR / "phase2_ep3"), is_trainable=True)

opt3   = AdamW([p for p in model3.parameters() if p.requires_grad], lr=PHASE3_LR, weight_decay=0.01)
steps3 = math.ceil(len(trainval_loader) / GRAD_ACCUM) * PHASE3_EPOCHS
warm3  = int(0.05 * steps3)   # shorter warmup since we're already well-trained
sched3 = get_cosine_schedule_with_warmup(opt3, warm3, steps3)

for epoch in range(PHASE3_EPOCHS):
    model3.train()
    total_loss3, n_steps3 = 0.0, 0
    opt3.zero_grad()
    for step, batch in enumerate(trainval_loader):
        batch = {k: v.to(model3.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        loss  = model3(**batch).loss / GRAD_ACCUM
        loss.backward()
        total_loss3 += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(trainval_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model3.parameters() if p.requires_grad], 1.0
            )
            opt3.step(); sched3.step(); opt3.zero_grad()
            n_steps3 += 1
        if (step + 1) % 100 == 0:
            print(f"  Phase3 Ep{epoch+1} | step {step+1}/{len(trainval_loader)} "
                  f"| loss {total_loss3/n_steps3:.4f} | lr {sched3.get_last_lr()[0]:.2e}")
    avg = total_loss3 / max(n_steps3, 1)
    print(f"Phase 3 Epoch {epoch+1}/{PHASE3_EPOCHS} done | avg loss {avg:.4f}")

    ep_ckpt = DRIVE_DIR / f"phase3_ep{epoch+1}"
    model3.save_pretrained(str(ep_ckpt))
    processor.save_pretrained(str(ep_ckpt))
    print(f"  Saved Phase 3 Epoch {epoch+1} → {ep_ckpt}")

del model3, base3
torch.cuda.empty_cache()
print("Phase 3 complete.")

In [10]:
# ── Cell 11: Ensemble inference across all Phase 1 + Phase 2 checkpoints ──────
from peft import PeftModel

ckpt_paths = [
    CKPT_DIR,                    # Phase 1 best (Epoch 2, val acc 0.6660)
    DRIVE_DIR / "phase2_ep1",
    DRIVE_DIR / "phase2_ep2",
    DRIVE_DIR / "phase2_ep3",    # same as final_ckpt
]

all_scores = []  # list of per-checkpoint score lists: [[n_choices floats] x n_test]

for ckpt_idx, ckpt in enumerate(ckpt_paths):
    print(f"\nCheckpoint {ckpt_idx+1}/{len(ckpt_paths)}: {ckpt.name}")
    base = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, quantization_config=bnb_cfg, device_map="auto", low_cpu_mem_usage=True,
    )
    m = PeftModel.from_pretrained(base, str(ckpt))
    m.eval()

    ckpt_scores = []
    for i, sample in enumerate(test_ds):
        enc = processor(text=[sample["prompt"]], images=[sample["image"]], return_tensors="pt")
        enc = {k: v.to(m.device) for k, v in enc.items()}
        with torch.no_grad():
            logits = m(**enc).logits[0, -1, :]
        s = [
            logits[processor.tokenizer.encode(
                f" {CHOICE_LETTERS[j]}", add_special_tokens=False
            )[-1]].item()
            for j in range(len(sample["choices"]))
        ]
        ckpt_scores.append(s)
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(test_ds)}")

    all_scores.append(ckpt_scores)
    del m, base
    torch.cuda.empty_cache()
    print(f"  Done.")

# Average log-likelihood scores across checkpoints, then argmax
ids, preds = [], []
for i in range(len(test_ds)):
    n = len(test_ds[i]["choices"])
    avg = np.mean([[all_scores[c][i][j] for j in range(n)] for c in range(len(ckpt_paths))], axis=0)
    ids.append(test_ds[i]["id"])
    preds.append(int(np.argmax(avg)))

print(f"\nEnsemble complete: {len(preds)} predictions from {len(ckpt_paths)} checkpoints.")

# Save submission
submission = pd.DataFrame({"id": ids, "answer": preds})
SUB_PATH = DRIVE_DIR / "submission.csv"
submission.to_csv(SUB_PATH, index=False)
print(f"submission.csv saved to: {SUB_PATH}")
print(submission.head(10))

assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(test_df)
assert submission["answer"].dtype in ["int64", "int32"]
assert (submission["answer"] >= 0).all()
print("\nAll checks passed. Ready to upload to Kaggle!")


Checkpoint 1/4: best_ckpt
  200/1008
  400/1008
  600/1008
  800/1008
  1000/1008
  Done.

Checkpoint 2/4: phase2_ep1
  200/1008
  400/1008
  600/1008
  800/1008
  1000/1008
  Done.

Checkpoint 3/4: phase2_ep2
  200/1008
  400/1008
  600/1008
  800/1008
  1000/1008
  Done.

Checkpoint 4/4: phase2_ep3
  200/1008
  400/1008
  600/1008
  800/1008
  1000/1008
  Done.

Ensemble complete: 1008 predictions from 4 checkpoints.
submission.csv saved to: /content/drive/MyDrive/smolvlm_competition/submission.csv
           id  answer
0  test_01750       2
1  test_00128       1
2  test_02891       3
3  test_02425       1
4  test_00930       2
5  test_03725       0
6  test_00009       0
7  test_02880       1
8  test_01208       1
9  test_00619       0

All checks passed. Ready to upload to Kaggle!


In [11]:
# ── Cell 12: Download submission.csv directly from Colab ─────────────────────
from google.colab import files
files.download(str(SUB_PATH))
print("Downloading submission.csv... (allow popups in your browser if blocked)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>